# 02 - Vault TVL And Fee Time-Series Monitoring

This notebook refreshes and analyzes DeFiLlama TVL/fees snapshots for Lazy Summer and Summer.fi.
It is designed to support interactive, repeatable monitoring.

**Source class:** API_DERIVED (DeFiLlama)


## What this notebook does

1. Optionally pull fresh DeFiLlama snapshots.
2. Load latest TVL and fee series snapshots from disk.
3. Plot TVL trend and daily fee run-rate.
4. Compute a transparent fee-productivity summary.


In [ ]:
from pathlib import Path
import sys
import json
import warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

sns.set_theme(style="whitegrid")

cwd = Path.cwd().resolve()
root = None
for candidate in [cwd, *cwd.parents]:
    if (candidate / "notebooks" / "notebook_utils.py").exists():
        root = candidate
        break
if root is None:
    raise RuntimeError("Could not locate repository root containing notebooks/notebook_utils.py")

if root.as_posix() not in sys.path:
    sys.path.insert(0, root.as_posix())

from notebooks import notebook_utils as nbx
ROOT = nbx.setup_notebook_context()
print(f"Project root: {ROOT}")
print(f"Notebook run started (UTC): {nbx.utc_now_iso()}")


In [ ]:
REFRESH_DEFILLAMA = True

if REFRESH_DEFILLAMA:
    nbx.refresh_defillama_snapshots()

lazy_tvl_path = nbx.latest_defillama_snapshot("lazy-summer-protocol", "tvl")
summer_tvl_path = nbx.latest_defillama_snapshot("summer.fi", "tvl")
fees_path = nbx.latest_defillama_snapshot("lazy-summer-protocol", "fees")

if not all([lazy_tvl_path, summer_tvl_path, fees_path]):
    raise FileNotFoundError("Could not find required DeFiLlama snapshot files.")

print("Lazy TVL snapshot:", lazy_tvl_path)
print("Summer.fi TVL snapshot:", summer_tvl_path)
print("Lazy fees snapshot:", fees_path)


In [ ]:
lazy_payload = nbx.load_json(lazy_tvl_path)
summer_payload = nbx.load_json(summer_tvl_path)
fees_payload = nbx.load_json(fees_path)

lazy_tvl = nbx.protocol_tvl_df(lazy_payload)
summer_tvl = nbx.protocol_tvl_df(summer_payload)
fees_df = nbx.fee_series_df(fees_payload)
fees_roll = nbx.rolling_annualized_fees(fees_df, window_days=30)

summary = {
    "lazy_latest_tvl_usd": float(lazy_tvl["totalLiquidityUSD"].iloc[-1]) if not lazy_tvl.empty else None,
    "lazy_peak_tvl_usd": float(lazy_tvl["totalLiquidityUSD"].max()) if not lazy_tvl.empty else None,
    "summer_latest_tvl_usd": float(summer_tvl["totalLiquidityUSD"].iloc[-1]) if not summer_tvl.empty else None,
    "fees_30d_usd": float(fees_df["dailyFeesUSD"].tail(30).sum()) if len(fees_df) >= 30 else None,
    "fees_90d_usd": float(fees_df["dailyFeesUSD"].tail(90).sum()) if len(fees_df) >= 90 else None,
}

display(pd.DataFrame([summary]))


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=False, constrained_layout=True)

axes[0].plot(lazy_tvl["date"], lazy_tvl["totalLiquidityUSD"], label="Lazy Summer TVL", color="#0b5cab", linewidth=2)
axes[0].plot(summer_tvl["date"], summer_tvl["totalLiquidityUSD"], label="Summer.fi TVL", color="#9c6644", linewidth=1.8, alpha=0.8)
axes[0].set_title("TVL History")
axes[0].set_ylabel("USD")
axes[0].legend()

axes[1].bar(fees_df["date"], fees_df["dailyFeesUSD"], color="#80b1d3", alpha=0.65, label="Daily fees")
if "fees_30d_annualized" in fees_roll.columns:
    axes[1].plot(
        fees_roll["date"],
        fees_roll["fees_30d_annualized"],
        color="#1d3557",
        linewidth=2,
        label="30d annualized fees",
    )
axes[1].set_title("Lazy Summer Fee Series")
axes[1].set_ylabel("USD")
axes[1].legend()

plt.show()


In [ ]:
productivity = nbx.summarize_fee_productivity(lazy_tvl, fees_df, fee_window_days=90)
display(pd.DataFrame([productivity]))

if productivity.get("status") == "OK":
    display(Markdown(
        f"""
**90d fee-productivity (transparent formula)**
- Window: `{productivity['window_start']}` to `{productivity['window_end']}`
- Annualized fees: `{nbx.fmt_usd(productivity['fees_annualized_usd'])}`
- Average TVL: `{nbx.fmt_usd(productivity['avg_tvl_usd'])}`
- Implied fee rate: `{nbx.fmt_pct(productivity['implied_fee_rate'])}`
"""
    ))
else:
    display(Markdown(f"Fee productivity summary unavailable: {productivity.get('message', 'unknown error')}"))
